In [1]:
import os, sys

ENV = 'colab' if os.environ.get('COLAB_RELEASE_TAG') else \
      'kaggle' if os.path.exists('/kaggle/input') else 'local'

print(f'Environment: {ENV}')

Environment: colab


In [2]:
# Kaggle 認証 (Colab 環境のみ)
if ENV == 'colab':
    import os
    # Colab Secrets に KAGGLE_TOKEN を登録している場合:
    # from google.colab import userdata
    # os.environ['KAGGLE_TOKEN'] = userdata.get('KAGGLE_TOKEN')

    # 直接設定する場合 (kaggle.json 不要の新形式トークン):
    os.environ['KAGGLE_TOKEN'] = 'KAGGLE_TOKEN_REMOVED'
    print('Kaggle auth set')

Kaggle auth set


In [3]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0


In [54]:
import warnings
warnings.simplefilter('ignore')

In [55]:
import os
import sys
import subprocess

In [56]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [57]:
if ENV == 'kaggle':
    set_env(
        input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz',
        temp_dir='/kaggle/tmp/setup'
    )

In [58]:
if ENV == 'colab':
    import subprocess, sys, os, glob

    # kagglehub をインストール
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=True)
    import kagglehub

    # aimo-3-utils をダウンロード
    utils_path = kagglehub.dataset_download('kalyankkr/aimo-3-utils')
    print(f'Utils path: {utils_path}')

    # wheels ディレクトリから直接インストール
    wheels_dirs = glob.glob(f'{utils_path}/**/wheels', recursive=True)
    wheels_dir = wheels_dirs[0] if wheels_dirs else f'{utils_path}/wheels'
    print(f'Wheels dir: {wheels_dir}')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--no-index', '--find-links', wheels_dir,
        'unsloth', 'trl', 'vllm', 'openai_harmony',
    ], check=True)

    # kaggle_evaluation: 既にインストール済みか確認、なければ試みる
    try:
        import kaggle_evaluation
        print('kaggle_evaluation: already installed')
    except ImportError:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', 'kaggle_evaluation'],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print('kaggle_evaluation not on PyPI, trying wheels...')
            kg_eval_wheels = glob.glob(f'{wheels_dir}/kaggle_evaluation*.whl')
            if kg_eval_wheels:
                subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    kg_eval_wheels[0]], check=True)
            else:
                print('WARNING: kaggle_evaluation not found, may fail on import')
        else:
            print('kaggle_evaluation: installed from PyPI')

    # tiktoken パスを設定
    tiktoken_dirs = glob.glob(f'{utils_path}/**/tiktoken_encodings', recursive=True)
    tiktoken_path = tiktoken_dirs[0] if tiktoken_dirs else '/content/setup/tiktoken_encodings'
    print(f'Tiktoken path: {tiktoken_path}')

    # モデルのダウンロード
    MODEL_PATH = kagglehub.model_download('danielhanchen/gpt-oss-120b/transformers/default')
    print(f'Model path: {MODEL_PATH}')

    # テストデータ: /kaggle/input に既にある場合はそれを使う
    _comp_dir = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3'
    if os.path.exists(f'{_comp_dir}/test.csv'):
        TEST_CSV = f'{_comp_dir}/test.csv'
        print(f'Test CSV (cached): {TEST_CSV}')
    else:
        try:
            data_path = kagglehub.competition_download('ai-mathematical-olympiad-progress-prize-3')
            TEST_CSV = f'{data_path}/test.csv'
        except Exception as e:
            print(f'competition_download failed: {e}')
            # フォールバック: リポジトリの test.csv をインラインで作成
            TEST_CSV = '/content/test.csv'
            with open(TEST_CSV, 'w') as _f:
                _f.write('"id","problem"\n')
                _f.write('"000aaa","What is $1-1$?"\n')
                _f.write('"111bbb","What is $0\\times10$?"\n')
                _f.write('"222ccc","Solve $4+x=4$ for $x$."\n')
            print(f'Test CSV created inline: {TEST_CSV}')
    print(f'Test CSV: {TEST_CSV}')

    # 50problems データセットのダウンロード
    problems_path = kagglehub.dataset_download('amanatar/50problems')
    print(f'50problems path: {problems_path}')

    # test.csv が存在しない場合は必ず作成
    if not os.path.exists(TEST_CSV):
        with open(TEST_CSV, 'w') as _f:
            _f.write('"id","problem"\n')
            _f.write('"000aaa","What is $1-1$?"\n')
            _f.write('"111bbb","What is $0\\times10$?"\n')
            _f.write('"222ccc","Solve $4+x=4$ for $x$."\n')
        print(f'Test CSV created: {TEST_CSV}')

    os.environ['TIKTOKEN_ENCODINGS_BASE'] = tiktoken_path
    os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'

else:  # kaggle
    MODEL_PATH = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    TEST_CSV = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv'
    os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'
    os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'

print(f'MODEL_PATH: {MODEL_PATH}')
print(f'TEST_CSV: {TEST_CSV}')

Using Colab cache for faster access to the 'aimo-3-utils' dataset.
Utils path: /kaggle/input/aimo-3-utils
Wheels dir: /kaggle/input/aimo-3-utils/wheels
kaggle_evaluation not on PyPI, trying wheels...
Tiktoken path: /kaggle/input/aimo-3-utils/tiktoken_encodings
Model path: /root/.cache/kagglehub/models/danielhanchen/gpt-oss-120b/transformers/default/1
competition_download failed: User is not authenticated
Test CSV created inline: /content/test.csv
Test CSV: /content/test.csv


100%|██████████| 4.55k/4.55k [00:00<00:00, 22.8MB/s]

Extracting files...
50problems path: /root/.cache/kagglehub/datasets/amanatar/50problems/versions/1
MODEL_PATH: /root/.cache/kagglehub/models/danielhanchen/gpt-oss-120b/transformers/default/1
TEST_CSV: /content/test.csv


In [59]:
if ENV == 'kaggle':
    subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

In [60]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# TRITON_PTXAS_PATH / TIKTOKEN_ENCODINGS_BASE はセットアップセルで設定済み

In [61]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

try:
    from openai_harmony import (
        HarmonyEncodingName,
        load_harmony_encoding,
        SystemContent,
        ReasoningEffort,
        ToolNamespaceConfig,
        Author,
        Message,
        Role,
        TextContent,
        Conversation,
    )
except ImportError as e:
    print(f'WARNING: openai_harmony not available: {e}')

from transformers import set_seed

try:
    import kaggle_evaluation.aimo_3_inference_server
except ImportError as e:
    print(f'WARNING: kaggle_evaluation not available: {e}')

In [62]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )

    ANSWER_ONLY_PROMPT = (
        'You are an IMO-level mathematician.'
        ' Think silently.'
        ' Do NOT explain.'
        ' Return only: \\boxed{number}'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )
    
    served_model_name = 'gpt-oss'
    model_path = MODEL_PATH
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 0.65
    min_p = 0.02

In [63]:
set_seed(CFG.seed)

In [64]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [65]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [66]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

In [72]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
                
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        return None
    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
    
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            
            if not isinstance(top_logprobs_dict, dict):
                continue
            
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count
    
    @staticmethod
    def _dynamic_timeout(deadline: float, cushion: float = 0.5, max_timeout: float = 30.0) -> float:
        if deadline is None:
            return max_timeout
        remaining = deadline - time.time()
        if remaining <= 0:
            return 0.0
        return min(max_timeout, max(0.2, remaining - cushion))

    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf'),
                'ReasoningText': ''
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        
        logprobs_buffer = []
        all_text_chunks = []
    
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )
    
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                req_timeout = self._dynamic_timeout(deadline, cushion=0.5, max_timeout=30.0)
                if req_timeout <= 0:
                    break

                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=self.cfg.temperature, 
                    logprobs=self.cfg.top_logprobs, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    timeout=req_timeout,
                    extra_body={
                        'min_p': self.cfg.min_p, 
                        'stop_token_ids': self.stop_token_ids, 
                        'return_token_ids': True
                    }
                )
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            all_text_chunks.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
    
                            if answer is not None:
                                final_answer = answer
                                break
    
                finally:
                    stream.close()
    
                if final_answer is not None:
                    break
    
                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    local_tool._local_jupyter_timeout = self._dynamic_timeout(deadline, cushion=1.0, max_timeout=15.0)
                    tool_responses = local_tool.process_sync_plus(last_message)
    
                    response_text = tool_responses[0].content[0].text
    
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1
    
                    conversation.messages.extend(tool_responses)
    
        except Exception as exc:
            print(f"[_process_attempt ERROR] attempt={attempt_index}: {type(exc).__name__}: {exc}")
            python_errors += 1
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
        reasoning_text = ''.join(all_text_chunks)
    
        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Entropy': mean_entropy, 
            'Answer': final_answer,
            'ReasoningText': reasoning_text
        }
    
    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer

    def _verify_answer(self, problem: str, answer: int) -> bool:

        verify_prompt = (
            f'Problem:\n{problem}\n\n'
            f'Proposed answer: {answer}\n\n'
            'Check the answer carefully.\n'
            'Reply with only ONE word: CORRECT or WRONG'
        )

        try:
            messages = self.template.apply_chat_template(
                self.cfg.ANSWER_ONLY_PROMPT,
                verify_prompt,
                AIMO3Tool(
                    local_jupyter_timeout=self.cfg.jupyter_timeout,
                    tool_prompt=self.cfg.tool_prompt
                ).tool_config
            )
            conversation = Conversation.from_messages(messages)
            prompt_ids = self.encoding.render_conversation_for_completion(
                conversation, Role.ASSISTANT
            )

            resp = self.client.completions.create(
                model=self.cfg.served_model_name,
                temperature=0.0,
                max_tokens=32,
                prompt=prompt_ids,
                extra_body={
                    'min_p': 0.0,
                    'stop_token_ids': self.stop_token_ids,
                    'return_token_ids': True
                }
            )

            text = resp.choices[0].text.strip().upper()
            return 'CORRECT' in text and 'WRONG' not in text

        except Exception:
            return False

    def _evolve_answer(self, problem: str, detailed_results: list) -> int:

        answer_counts = Counter(
            r['Answer'] for r in detailed_results if r['Answer'] is not None
        )
        top_candidates = answer_counts.most_common(3)

        if not top_candidates:
            return 0

        answer_to_shortest = {}
        for r in detailed_results:
            ans = r['Answer']
            text = r.get('ReasoningText', '')
            if ans is None or not text:
                continue
            if ans not in answer_to_shortest or len(text) < len(answer_to_shortest[ans]):
                answer_to_shortest[ans] = text

        solutions_block = ''
        for i, (ans, count) in enumerate(top_candidates):
            summary = answer_to_shortest.get(ans, '(no reasoning captured)')
            if len(summary) > 8000:
                summary = summary[:4000] + '\n... (truncated) ...\n' + summary[-4000:]
            solutions_block += (
                f'--- Solution {i} (answer={ans}, votes={count}) ---\n'
                f'{summary}\n\n'
            )

        evolve_prompt = (
            f'You will be given a challenging math problem followed by '
            f'{len(top_candidates)} candidate solutions. '
            f'The solutions may be correct or incorrect.\n\n'
            f'Your task is to carefully analyze each solution, identify errors, '
            f'and produce your own rigorous solution.\n\n'
            f'Problem:\n{problem}\n\n'
            f'Candidate solutions:\n{solutions_block}\n'
            f'Now produce your own detailed solution. '
            f'The final answer must be a non-negative integer between 0 and 99999.\n'
            f'Place your final answer inside \\boxed{{}}.\n'
        )

        print(f'\n[EVOLVE] Launching final-judge attempt with {len(top_candidates)} candidates...\n')

        try:
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout,
                tool_prompt=self.cfg.tool_prompt
            )

            messages = self.template.apply_chat_template(
                self.cfg.system_prompt,
                evolve_prompt,
                local_tool.tool_config
            )
            conversation = Conversation.from_messages(messages)

            final_answer = None

            for _ in range(self.cfg.turns):
                prompt_ids = self.encoding.render_conversation_for_completion(
                    conversation, Role.ASSISTANT
                )
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
                if max_tokens < self.cfg.buffer_tokens:
                    break

                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=0.0,
                    max_tokens=max_tokens,
                    prompt=prompt_ids,
                    seed=self.cfg.seed,
                    stream=True,
                    extra_body={
                        'min_p': 0.0,
                        'stop_token_ids': self.stop_token_ids,
                        'return_token_ids': True
                    }
                )

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text

                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            text_chunks.append(new_text)

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
                            if answer is not None:
                                final_answer = answer
                                break
                finally:
                    stream.close()

                if final_answer is not None:
                    break

                if not token_buffer:
                    break

                new_messages = self.encoding.parse_messages_from_completion_tokens(
                    token_buffer, Role.ASSISTANT
                )
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break

                if last_message.recipient == 'python':
                    tool_responses = local_tool.process_sync_plus(last_message)
                    conversation.messages.extend(tool_responses)

            if final_answer is not None:
                print(f'\n[EVOLVE] Final-judge answer: {final_answer}\n')
                return final_answer

        except Exception as exc:
            print(f'[EVOLVE] Error: {exc}')

        print('[EVOLVE] Failed to produce answer, falling back to entropy vote.')
        return self._select_answer(detailed_results)
    
    def solve_problem(self, problem: str) -> int:

        print(f'\nProblem: {problem}\n')
    
        user_input = f'{problem} {self.cfg.preference_prompt}'    
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
    
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
    
        tasks = []
        for attempt_index in range(self.cfg.attempts):
            if attempt_index < 4:
                system_prompt = self.cfg.ANSWER_ONLY_PROMPT
            else:
                system_prompt = self.cfg.system_prompt
            tasks.append((system_prompt, attempt_index))
    
        detailed_results = []
        valid_answers = []
    
        stop_event = threading.Event()
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
            for (system_prompt, attempt_index) in tasks:
                futures.append(
                    executor.submit(
                        self._process_attempt,
                        user_input,
                        system_prompt,
                        attempt_index,
                        stop_event,
                        deadline
                    )
                )
    
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
    
                    counts = Counter(valid_answers).most_common(1)
                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()
                        for f in futures:
                            f.cancel()
                        break
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)
    
        if detailed_results:
            df = pd.DataFrame(detailed_results)
            df['Entropy'] = df['Entropy'].round(3)
            df['Answer'] = df['Answer'].astype('Int64')
            display(df.drop(columns=['ReasoningText'], errors='ignore'))

        if not valid_answers:
            print('\nResult: 0\n')
            return 0

        if len(valid_answers) >= 4:
            most_common, count = Counter(valid_answers).most_common(1)[0]
            if count >= 4:
                print(f'\nUNANIMOUS ANSWER: {most_common}\n')
                return most_common

        answer_counts = Counter(valid_answers)
        candidates = [a for a, c in answer_counts.items() if c >= 2]

        entropy_map = {}
        for r in detailed_results:
            if r['Answer'] is not None and r['Entropy'] is not None:
                entropy_map.setdefault(r['Answer'], []).append(r['Entropy'])

        avg_entropy = {a: sum(v) / len(v) for a, v in entropy_map.items()}
        candidates = sorted(candidates, key=lambda x: avg_entropy.get(x, 999))

        for ans in candidates:
            try:
                if self._verify_answer(problem, ans):
                    print(f'\nVERIFIED ANSWER: {ans}\n')
                    return ans
            except Exception:
                pass

        return self._evolve_answer(problem, detailed_results)

    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass

In [73]:
solver = AIMO3Solver(CFG)

Loading model weights from /root/.cache/kagglehub/models/danielhanchen/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 3.05 seconds.

Waiting for vLLM server...
Server is ready (took 35.73 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 1.25 seconds.



In [74]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [75]:
if ENV == 'kaggle':
    # Kaggle: 公式の評価サーバーを使用
    inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
    if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        inference_server.serve()
    else:
        inference_server.run_local_gateway((TEST_CSV,))

else:
    # Colab: solver を直接呼び出して結果を確認
    import polars as pl
    test_df = pl.read_csv(TEST_CSV)
    results = []
    for row in test_df.iter_rows(named=True):
        answer = solver.solve_problem(row['problem'])
        results.append({'id': row['id'], 'answer': answer})
        print(f"[{row['id']}] {row['problem'][:60]} → {answer}")
    result_df = pl.DataFrame(results)
    print(result_df)



Problem: What is $1-1$?

Budget: 900.00 seconds | Deadline: 1775293329.06

[_process_attempt ERROR] attempt=6: HarmonyError: Unexpected EOS while waiting for message header to complete
[_process_attempt ERROR] attempt=4: HarmonyError: Unexpected EOS while waiting for message header to complete
[_process_attempt ERROR] attempt=7: HarmonyError: Unexpected EOS while waiting for message header to complete
[_process_attempt ERROR] attempt=5: HarmonyError: Unexpected EOS while waiting for message header to complete


,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,4,96,0,0,0.690,0
1,1,111,0,0,0.549,0
2,2,114,0,0,0.803,0
3,3,122,0,0,0.848,0



UNANIMOUS ANSWER: 0

[000aaa] What is $1-1$? → 0

Problem: What is $0\times10$?

Budget: 900.00 seconds | Deadline: 1775293334.36

[_process_attempt ERROR] attempt=5: HarmonyError: Unexpected EOS while waiting for message header to complete
[_process_attempt ERROR] attempt=4: HarmonyError: Unexpected EOS while waiting for message header to complete
[_process_attempt ERROR] attempt=6: HarmonyError: Unexpected EOS while waiting for message header to complete


,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,1,77,0,0,0.692,0
1,2,78,0,0,0.621,0
2,4,83,0,0,0.735,0
3,3,109,0,0,0.774,0



UNANIMOUS ANSWER: 0

[111bbb] What is $0\times10$? → 0

Problem: Solve $4+x=4$ for $x$.

Budget: 900.00 seconds | Deadline: 1775293336.64

[_process_attempt ERROR] attempt=5: HarmonyError: Unexpected EOS while waiting for message header to complete
[_process_attempt ERROR] attempt=7: HarmonyError: Unexpected EOS while waiting for message header to complete
[_process_attempt ERROR] attempt=6: HarmonyError: Unexpected EOS while waiting for message header to complete
[_process_attempt ERROR] attempt=4: HarmonyError: Unexpected EOS while waiting for message header to complete


,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,2,73,0,0,0.657,0
1,4,94,0,0,0.726,0
2,1,99,0,0,0.797,0
3,3,111,0,0,0.773,0



UNANIMOUS ANSWER: 0

[222ccc] Solve $4+x=4$ for $x$. → 0
shape: (3, 2)
┌────────┬────────┐
│ id     ┆ answer │
│ ---    ┆ ---    │
│ str    ┆ i64    │
╞════════╪════════╡
│ 000aaa ┆ 0      │
│ 111bbb ┆ 0      │
│ 222ccc ┆ 0      │
└────────┴────────┘


In [ ]:
import polars as pl
import pandas as pd
import os

# 50問テスト用 CSV のパス (環境に応じて切り替え)
import glob as _glob
_candidates = [
    '/kaggle/input/50problems/50problems.csv',
    *_glob.glob('/root/.cache/kagglehub/datasets/amanatar/50problems/**/50problems.csv', recursive=True),
    'data/50problems/50problems.csv',
]
FILE_PATH = next((p for p in _candidates if os.path.exists(p)), None)
print(f'FILE_PATH: {FILE_PATH}')

# --- Predict 関数 (kaggle_evaluation 不要版) ---
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer=None) -> pl.DataFrame:
    id_value = id_[0, 0]
    question_text = question[0, 0]
    gc.disable()
    final_answer = solver.solve_problem(question_text)
    gc.enable()
    gc.collect()
    return pl.DataFrame({'id': [id_value], 'answer': [final_answer]})

# --- テストループ ---
if not os.path.exists(FILE_PATH):
    print(f'Error: File not found at {FILE_PATH}')
else:
    external_df = pd.read_csv(FILE_PATH)
    test_results = []
    print(f'Starting test on {len(external_df)} problems...\n')

    for idx, row in external_df.iterrows():
        problem_text = row['Problem']
        ground_truth = row['Answer']

        print(f"{'='*50}")
        print(f'TESTING PROBLEM {idx+1}')
        print(f'Statement: {problem_text[:80]}...')
        print(f'Ground Truth: {ground_truth}')
        print(f"{'-'*50}")

        id_df = pl.DataFrame({'id': [f'ext_{idx}']})
        question_df = pl.DataFrame({'question': [problem_text]})

        try:
            result_pl_df = predict(id_df, question_df)
            predicted_val = result_pl_df[0, 'answer']
            is_correct = (int(predicted_val) == int(ground_truth))
            test_results.append({'idx': idx+1, 'prediction': predicted_val,
                                 'ground_truth': ground_truth, 'correct': is_correct})
            status = '✅ CORRECT' if is_correct else '❌ INCORRECT'
            print(f'\n[Problem {idx+1}] Predicted: {predicted_val} | {status}\n')
        except Exception as e:
            print(f'Error on problem {idx+1}: {e}')
            test_results.append({'idx': idx+1, 'prediction': None,
                                 'ground_truth': ground_truth, 'correct': False})

    summary_df = pd.DataFrame(test_results)
    display(summary_df)
    print(f'Overall Accuracy: {summary_df["correct"].mean() * 100:.2f}%')


FILE_PATH: /root/.cache/kagglehub/datasets/amanatar/50problems/versions/1/50problems.csv
Starting test on 50 problems...

TESTING PROBLEM 1
Statement: Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Poi...
Ground Truth: 336
--------------------------------------------------

Problem: Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie on segments $BC$ and $AC$, respectively, such that $AD=AE=AB$. Line $DE$ intersects $AB$ at $X$. Circles $BXD$ and $CED$ intersect for the second time at $Y \neq D$. Suppose that $Y$ lies on line $AD$. There is a unique such triangle with minimal perimeter. This triangle has side lengths $a=BC$, $b=CA$, and $c=AB$. Find the remainder when $abc$ is divided by $10^{5}$.

Budget: 900.00 seconds | Deadline: 1775293339.24

[_process_attempt ERROR] attempt=7: HarmonyError: Unexpected EOS while waiting for message header to complete
[_process_attempt ERROR] attempt=6: HarmonyError: Unex

,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,4,10967,7,0,0.566,336
1,3,11277,10,1,0.609,336
2,1,11378,3,0,0.525,336
3,6,12403,17,0,0.620,336



UNANIMOUS ANSWER: 336


[Problem 1] Predicted: 336 | ✅ CORRECT

TESTING PROBLEM 2
Statement: Define a function $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ by $f(n...
Ground Truth: 32951
--------------------------------------------------

Problem: Define a function $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ by $f(n) = \sum_{i = 1}^n \sum_{j = 1}^n j^{1024} \lfloor\frac1j + \frac{n-i}{n}\rfloor$. Let $M=2 \cdot 3 \cdot 5 \cdot 7 \cdot 11 \cdot 13$ and let $N = f(M^{15}) - f(M^{15}-1)$. Let $k$ be the largest non-negative integer such that $2^k$ divides $N$. What is the remainder when $2^k$ is divided by $5^7$?

Budget: 900.00 seconds | Deadline: 1775293517.26

[_process_attempt ERROR] attempt=5: HarmonyError: Unexpected EOS while waiting for message header to complete


,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,7,3834,3,0,0.548,32951
1,5,4752,8,1,0.563,32951
2,4,5217,1,0,0.592,32951
3,3,5548,3,0,0.555,32951



UNANIMOUS ANSWER: 32951


[Problem 2] Predicted: 32951 | ✅ CORRECT

TESTING PROBLEM 3
Statement: A tournament is held with $2^{20}$ runners each of which has a different running...
Ground Truth: 21818
--------------------------------------------------

Problem: A tournament is held with $2^{20}$ runners each of which has a different running speed. The competition consists of $20$ rounds. The winner of each race in the $i^{\text{th}}$ round receives $2^{20-i}$ points and the loser gets no points. Let $N$ denote the number of possible orderings of the competitors at the end of the tournament. Let $k$ be the largest positive integer such that $10^k$ divides $N$. What is the remainder when $k$ is divided by $10^{5}$?

Budget: 900.00 seconds | Deadline: 1775293590.82



...


UNANIMOUS ANSWER: 50


[Problem 7] Predicted: 50 | ✅ CORRECT

TESTING PROBLEM 8
Statement: Let $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ satisfy $f(m) + f(n) ...
Ground Truth: 580
--------------------------------------------------

Problem: Let $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ satisfy $f(m) + f(n) = f(m + n + mn)$ for all $m, n$. Across all functions where $f(n) \leq 1000$ for all $n \leq 1000$, how many different values can $f(2024)$ take?

Budget: 900.00 seconds | Deadline: 1775296490.25

[_process_attempt ERROR] attempt=5: HarmonyError: Unexpected EOS while waiting for message header to complete


,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,1,6465,13,0,0.748,580
1,2,8774,9,1,0.740,580
2,3,10643,12,1,0.785,580
3,8,11586,12,4,0.686,580



UNANIMOUS ANSWER: 580


[Problem 8] Predicted: 580 | ✅ CORRECT

TESTING PROBLEM 9
Statement: A $500 \times 500$ square is divided into $k$ rectangles with integer side lengt...
Ground Truth: 520
--------------------------------------------------

Problem: A $500 \times 500$ square is divided into $k$ rectangles with integer side lengths. Given that no two of these rectangles have the same perimeter, the largest possible value of $k$ is $\mathcal{K}$. What is the remainder when $\mathcal{K}$ is divided by $10^{5}$?

Budget: 900.00 seconds | Deadline: 1775296653.16

[_process_attempt ERROR] attempt=7: HarmonyError: Unexpected EOS while waiting for message header to complete


,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,3,7177,2,0,0.866,520
1,2,27728,11,0,0.843,520
2,1,29907,15,0,0.839,520
3,4,35701,18,2,0.836,706
4,5,39633,10,0,0.860,520



UNANIMOUS ANSWER: 520


[Problem 9] Predicted: 520 | ✅ CORRECT

TESTING PROBLEM 10
Statement: Let $\mathcal{F}$ be the set of functions $\alpha \colon \mathbb{Z} \to \mathbb{...
Ground Truth: 160
--------------------------------------------------

Problem: Let $\mathcal{F}$ be the set of functions $\alpha \colon \mathbb{Z} \to \mathbb{Z}$ with finite support. Define a product $\alpha \star \beta = \sum \alpha(n) \beta(n)$. A function is shifty if $\alpha(m)=0$ for $m<0, m>8$ and there exists $\beta$ such that $S_n(\alpha) \star \beta = 1$ for two distinct shifts and $0$ otherwise. How many shifty functions are there?

Budget: 900.00 seconds | Deadline: 1775297248.85

